# DengAI · 1. Problem statement and data

**Goal:** forecast the number of dengue cases reported each week in
**San Juan (Puerto Rico)** and **Iquitos (Peru)** from environmental data.

| Notebook | Rubric item |
|---|---|
| **01 — Problem and data** (this one) | Problem & motivation (4 pts) · Load data, split X / y (2 pts) |
| 02 — Exploratory analysis | Seaborn EDA and findings (4 pts) |
| 03 — Preprocessing and features | Encoding, missing values, features, target (5 pts) |
| 04 — Validation and model selection | CV design, metric, models, hyperparameters (7 pts) |
| 05 — Results and conclusions | Analysis, applicability, impact (8 pts) |

## Why this problem matters

Dengue is a mosquito-borne viral disease (transmitted mainly by *Aedes aegypti*).
The WHO estimates that around half of the world's population is at risk, with
roughly 100–400 million infections per year. There is no specific treatment;
severe dengue requires timely hospital care.

Outbreaks are **seasonal but irregular**: some years are mild and others
overwhelm clinics. Mosquito breeding and viral replication depend on rainfall,
humidity and temperature, so climate is a plausible *leading indicator* of
cases.

**How a model helps.** A forecast of weekly case counts lets public-health
teams act *before* cases peak instead of reacting after an outbreak is visible:

* schedule vector control (breeding-site removal, larvicide, fogging) for the
  weeks when transmission is expected to rise;
* plan hospital beds, staff, IV fluids and diagnostic tests;
* time public awareness campaigns.

Every avoided case reduces medical costs and lost work days, and earlier
action is cheaper than emergency response.

**Competition framing** ([DrivenData: *DengAI — Predicting Disease Spread*](https://www.drivendata.org/competitions/44/dengai-predicting-disease-spread/)):
predict `total_cases` for every test week; the score is the
**mean absolute error (MAE)** in cases per week. The test period is a
*multi-year* block after the training data, and **case counts are never
available during it**, so the model must rely on climate and season.

## Load the data with pandas

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dengai").is_dir())
sys.path.insert(0, str(ROOT))
OUTPUT_DIR = ROOT / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd

from dengai.config import CITY_NAMES, DATE_COLUMN, KEY_COLUMNS, TARGET_COLUMN
from dengai.data import find_data_dir, load_data, split_xy

pd.set_option("display.max_columns", 30)
DATA_DIR = find_data_dir(ROOT)
sorted(p.name for p in DATA_DIR.glob("*.csv"))

['dengue_features_test.csv',
 'dengue_features_train.csv',
 'dengue_labels_train.csv',
 'submission_format.csv']

The training features and labels are separate files keyed by
`(city, year, weekofyear)`. `load_data` merges them one-to-one and checks
that keys are unique, that the test rows match the submission template, and
that each city's test weeks start the week after its training data ends.

In [2]:
train, test, template = load_data(DATA_DIR)
print("train:", train.shape, " test:", test.shape, " template:", template.shape)
train.head()

train: (1456, 25)  test: (416, 24)  template: (416, 4)


,city,year,weekofyear,week_start_date,ndvi_ne,ndvi_nw,ndvi_se,ndvi_sw,precipitation_amt_mm,reanalysis_air_temp_k,reanalysis_avg_temp_k,reanalysis_dew_point_temp_k,reanalysis_max_air_temp_k,reanalysis_min_air_temp_k,reanalysis_precip_amt_kg_per_m2,reanalysis_relative_humidity_percent,reanalysis_sat_precip_amt_mm,reanalysis_specific_humidity_g_per_kg,reanalysis_tdtr_k,station_avg_temp_c,station_diur_temp_rng_c,station_max_temp_c,station_min_temp_c,station_precip_mm,total_cases
0,sj,1990,18,1990-04-30,0.122600,0.103725,0.198483,0.177617,12.42,297.572857,297.742857,292.414286,299.8,295.9,32.00,73.365714,12.42,14.012857,2.628571,25.442857,6.900000,29.4,20.0,16.0,4
1,sj,1990,19,1990-05-07,0.169900,0.142175,0.162357,0.155486,22.82,298.211429,298.442857,293.951429,300.9,296.4,17.94,77.368571,22.82,15.372857,2.371429,26.714286,6.371429,31.7,22.2,8.6,5
2,sj,1990,20,1990-05-14,0.032250,0.172967,0.157200,0.170843,34.54,298.781429,298.878571,295.434286,300.5,297.3,26.10,82.052857,34.54,16.848571,2.300000,26.714286,6.485714,32.2,22.8,41.4,4
3,sj,1990,21,1990-05-21,0.128633,0.245067,0.227557,0.235886,15.36,298.987143,299.228571,295.310000,301.4,297.0,13.90,80.337143,15.36,16.672857,2.428571,27.471429,6.771429,33.3,23.3,4.0,3
4,sj,1990,22,1990-05-28,0.196200,0.262200,0.251200,0.247340,7.52,299.518571,299.664286,295.821429,301.9,297.5,12.20,80.460000,7.52,17.210000,3.014286,28.942857,9.371429,35.0,23.9,5.8,6


In [3]:
periods = pd.concat(
    [
        frame.groupby("city")[DATE_COLUMN].agg(first="min", last="max", weeks="size").assign(split=split)
        for split, frame in [("train", train), ("test", test)]
    ]
).reset_index()
periods["city"] = periods["city"].map(CITY_NAMES)
periods.pivot(index="city", columns="split", values=["first", "last", "weeks"])

first                  last            weeks      
split          test      train       test      train  test train
city                                                            
Iquitos  2010-07-02 2000-07-01 2013-06-25 2010-06-25   156   520
San Juan 2008-04-29 1990-04-30 2013-04-23 2008-04-22   260   936

San Juan has 18 years of labelled weeks and must be forecast 5 years ahead;
Iquitos has 10 years and a 3-year horizon. Both horizons are long, which
rules out models that need last week's case count.

## What the columns contain

In [4]:
groups = {
    "keys": KEY_COLUMNS + [DATE_COLUMN],
    "vegetation index (satellite NDVI)": [c for c in train if c.startswith("ndvi")],
    "satellite rainfall (PERSIANN)": ["precipitation_amt_mm"],
    "NOAA reanalysis climate": [c for c in train if c.startswith("reanalysis")],
    "local weather station (GHCN)": [c for c in train if c.startswith("station")],
    "target": [TARGET_COLUMN],
}
pd.DataFrame(
    [(group, len(cols), ", ".join(cols)) for group, cols in groups.items()],
    columns=["group", "n", "columns"],
)

,group,n,columns
0,keys,4,"city, year, weekofyear, week_start_date"
1,vegetation index (satellite NDVI),4,"ndvi_ne, ndvi_nw, ndvi_se, ndvi_sw"
2,satellite rainfall (PERSIANN),1,precipitation_amt_mm
3,NOAA reanalysis climate,10,"reanalysis_air_temp_k, reanalysis_avg_temp_k, ..."
4,local weather station (GHCN),5,"station_avg_temp_c, station_diur_temp_rng_c, s..."
5,target,1,total_cases


In [5]:
train.dtypes.value_counts()

float64           20
int64              3
str                1
datetime64[us]     1
Name: count, dtype: int64

All 20 climate inputs are numeric. The only non-numeric fields are `city`
(categorical, 2 values) and `week_start_date` (a date). How they are
encoded is decided in notebook 03.

## Split into inputs X and output y

`y` is the weekly case count `total_cases`. `X` contains everything else,
including the keys: `city` selects the model, and the date and week
number are used for ordering and seasonality. The test set has the same
X columns and no y.

In [6]:
X, y = split_xy(train)
X_test = test.copy()

assert list(X.columns) == list(X_test.columns)
assert TARGET_COLUMN not in X_test
print("X:", X.shape, " y:", y.shape, " X_test:", X_test.shape)
y.describe().round(1)

X: (1456, 24)  y: (1456,)  X_test: (416, 24)


count    1456.0
mean       24.7
std        43.6
min         0.0
25%         5.0
50%        12.0
75%        28.0
max       461.0
Name: total_cases, dtype: float64

Because the two cities differ in scale and climate (see notebook 02), every
model is fitted per city. The per-city views of X and y are:

In [7]:
for city, name in CITY_NAMES.items():
    rows = X["city"].eq(city)
    print(f"{name:9s}  X={X[rows].shape}  y={y[rows].shape}  "
          f"X_test={X_test[X_test['city'].eq(city)].shape}  "
          f"mean cases/week={y[rows].mean():.1f}")

San Juan   X=(936, 24)  y=(936,)  X_test=(260, 24)  mean cases/week=34.2
Iquitos    X=(520, 24)  y=(520,)  X_test=(156, 24)  mean cases/week=7.6


### Summary

* 1,456 labelled weeks (936 San Juan + 520 Iquitos) with 20 climate inputs;
  416 test weeks to forecast (260 + 156).
* The target is a non-negative weekly count. The metric is MAE.
* Test weeks follow the training weeks directly, but test case counts are
  unknown for 3–5 years, so the inputs must be climate and calendar only.